# RegComp — GPU experiments (Colab)

Runs the parts of the paper that need Hugging Face models: MahaBERT / IndicBERT / XLM-R / multilingual mpnet linking, contrastive fine-tuning, the fine-tuned obligation classifiers (after annotation) and flan-T5 simplification.

Runtime → Change runtime type → **GPU** (T4 is enough). The results land in `conference-method/results/*_gpu.md`; download them or commit them.

In [ ]:
!git clone --depth 1 -b claude/peaceful-clarke-di2hty https://github.com/KulkarniShripad/RBI-reg-complience.git
%cd RBI-reg-complience/conference-method
!pip -q install -r requirements-gpu.txt

In [ ]:
# data: ingest the circulars in the repo, generate the synthetic evidence and the monitoring data (~6 min)
!python -m regcomp.ingest --pdfs ../circulars --out work/corpus
!python -m synth.generate --out work/synth --seed 7
!python -m synth.anomaly --out work/anomaly

In [ ]:
# local bge-small ONNX copy (for the bge-onnx rows)
!pip -q install -U huggingface_hub
!huggingface-cli download Xenova/bge-small-en-v1.5 tokenizer.json onnx/model.onnx --local-dir models/bge-small-en-v1.5
import os; os.environ['BGE_ONNX_DIR'] = 'models/bge-small-en-v1.5'

In [ ]:
# linking and verification with every encoder of the paper
!PYTHONPATH=. python experiments/eval_linking.py --encoders tfidf,bge-onnx,mahabert,indicbert,xlmr,mpnet --tag gpu
!PYTHONPATH=. python experiments/eval_verification.py --encoders tfidf,bge-onnx,mahabert,indicbert,xlmr,mpnet --tag gpu

In [ ]:
# contrastive fine-tuning on clause-evidence pairs (3 epochs, 1e-5, bs 16), then evaluate the fine-tuned linkers
for base in ['mpnet', 'mahabert', 'indicbert', 'xlmr']:
    !PYTHONPATH=. python experiments/finetune_contrastive.py --base {base}
!PYTHONPATH=. python experiments/eval_linking.py --encoders st:work/models/contrastive-mpnet,st:work/models/contrastive-mahabert,st:work/models/contrastive-indicbert,st:work/models/contrastive-xlmr --tag ft
!PYTHONPATH=. python experiments/eval_verification.py --encoders st:work/models/contrastive-mpnet,st:work/models/contrastive-mahabert,st:work/models/contrastive-indicbert,st:work/models/contrastive-xlmr --tag ft --no-baselines

In [ ]:
# flan-T5 simplification with the guard
!PYTHONPATH=. python experiments/eval_simplification.py --t5 google/flan-t5-base

In [ ]:
# after annotation (annotation/obligation_gold.csv exists): fine-tuned obligation classifiers, 5-fold CV x 3 seeds
!PYTHONPATH=. python experiments/finetune_classifier.py --models mahabert,indicbert,xlmr
!PYTHONPATH=. python experiments/eval_obligations.py

In [ ]:
!PYTHONPATH=. python experiments/bench_scalability.py --encoders tfidf,bge-onnx,mahabert,xlmr
!zip -r results.zip results annotation/*.csv
from google.colab import files; files.download('results.zip')